# Week 4 — Data Science Internship Project
This notebook contains dataset understanding, cleaning, EDA, visualizations, and a baseline predictive model.
Place your CSV dataset inside the `data/` folder as `dataset.csv`.

In [ ]:
# Import libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-whitegrid')

## 1) Dataset Understanding
Load the dataset and print basic information to understand its structure.

In [ ]:
# Set dataset path (change if your file is named differently)
DATA_PATH = os.path.join('..','data','dataset.csv') if os.path.exists(os.path.join('..','data','dataset.csv')) else 'data/dataset.csv'
print('Looking for dataset at:', DATA_PATH)

# Load dataset
try:
    df = pd.read_csv(DATA_PATH)
    print('Dataset loaded successfully')
except FileNotFoundError:
    raise FileNotFoundError('Please place your CSV file at data/dataset.csv')

# Show first 5 rows
print('
First 5 rows:')
display(df.head())

# Show dataset shape
print('Dataset shape:', df.shape)

# Show column names and data types
print('
Columns:')
print(df.columns.tolist())

print('
Data types:')
print(df.dtypes)

# Show missing values per column
print('
Missing values (per column):')
print(df.isnull().sum())

# Summary statistics for numeric columns
print('
Summary statistics:')
display(df.describe(include='all'))

## 2) Data Cleaning & Preprocessing
This section performs common cleaning steps: missing values, duplicates, types, outliers, and column name cleanup. All operations are commented for beginners.

In [ ]:
# Make a copy to avoid modifying original dataframe in-memory during exploration
df_clean = df.copy()

# 1. Clean column names: strip whitespace, lowercase, replace spaces with underscores
df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_')
print('Cleaned column names:')
print(df_clean.columns.tolist())

# 2. Remove exact duplicate rows if any
n_duplicates = df_clean.duplicated().sum()
print(f'Found {n_duplicates} duplicate rows')
if n_duplicates > 0:
    df_clean = df_clean.drop_duplicates().reset_index(drop=True)
    print('Dropped duplicates. New shape:', df_clean.shape)

# 3. Convert obvious date columns to datetime (heuristic: column name contains 'date')
for col in df_clean.columns:
    if 'date' in col:
        try:
            df_clean[col] = pd.to_datetime(df_clean[col])
            print(f'Converted {col} to datetime')
        except Exception:
            pass

# 4. Handle missing values: numeric -> median, categorical -> mode
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_clean.select_dtypes(exclude=[np.number, 'datetime']).columns.tolist()

# Fill numeric missing with median
for col in numeric_cols:
    if df_clean[col].isnull().any():
        med = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(med)
        print(f'Filled missing numeric values in {col} with median = {med}')

# Fill categorical missing with mode
for col in cat_cols:
    if df_clean[col].isnull().any():
        mode = df_clean[col].mode(dropna=True)
        if not mode.empty:
            df_clean[col] = df_clean[col].fillna(mode[0])
            print(f'Filled missing categorical values in {col} with mode = {mode[0]}')

# 5. Detect outliers using IQR method for numeric columns (mark them)
outliers = {}
for col in numeric_cols:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    is_out = ((df_clean[col] < lower) | (df_clean[col] > upper))
    count_out = is_out.sum()
    if count_out > 0:
        outliers[col] = int(count_out)

print('
Outlier counts by numeric column:')
print(outliers)

# After inspection, you may remove or cap outliers. Here we will keep them but provide a way to cap if desired.
# Example: cap values outside the 1st/99th percentiles (uncomment to apply)
# for col in numeric_cols:
#     lower = df_clean[col].quantile(0.01)
#     upper = df_clean[col].quantile(0.99)
#     df_clean[col] = df_clean[col].clip(lower, upper)

# Final shape after cleaning
print('
Final cleaned shape:', df_clean.shape)

## 3) Exploratory Data Analysis (EDA)
Use visualizations and summary statistics to understand relationships and distributions.

In [ ]:
# Identify numeric and categorical columns for EDA
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_clean.select_dtypes(exclude=[np.number, 'datetime']).columns.tolist()
print('Numeric columns:', num_cols)
print('Categorical columns:', cat_cols)

# 1. Correlation analysis (numeric features)
if len(num_cols) >= 2:
    corr = df_clean[num_cols].corr()
    print('
Correlation matrix:')
    display(corr)

    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
    plt.title('Feature Correlation Heatmap')
    plt.show()

# 2. Distribution plots and histograms for numeric columns
for col in num_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_clean[col], kde=True, bins=30)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.show()

# 3. Boxplots to check spread and outliers
for col in num_cols:
    plt.figure(figsize=(6, 3))
    sns.boxplot(x=df_clean[col])
    plt.title(f'Boxplot of {col}')
    plt.show()

# 4. Count plots for categorical columns
for col in cat_cols[:5]:  # limit to first 5 to avoid too many plots
    plt.figure(figsize=(8, 4))
    sns.countplot(y=df_clean[col], order=df_clean[col].value_counts().index)
    plt.title(f'Count plot of {col}')
    plt.show()

# 5. Trend analysis example: if a datetime column exists, plot trend of a numeric column over time
date_cols = [c for c in df_clean.columns if 'date' in c]
if date_cols and num_cols:
    dt = date_cols[0]
    metric = num_cols[0]
    ts = df_clean.set_index(dt).resample('M')[metric].mean().dropna()
    plt.figure(figsize=(10, 4))
    ts.plot(marker='o')
    plt.title(f'Trend of {metric} over time ({dt})')
    plt.ylabel(metric)
    plt.xlabel(dt)
    plt.show()

## 4) Data Visualization (save plots to visuals/)
Create common plot types and save them into the `visuals/` folder with short insights.

In [ ]:
os.makedirs('visuals', exist_ok=True)

# Example: Bar chart for the first categorical column (if exists)
if cat_cols:
    col = cat_cols[0]
    plt.figure(figsize=(8, 5))
    order = df_clean[col].value_counts().index
    sns.barplot(x=df_clean[col].value_counts().values, y=order, palette='viridis')
    plt.title(f'Bar chart of {col}')
    plt.xlabel('Count')
    plt.ylabel(col)
    fname = f'visuals/bar_{col}.png'
    plt.tight_layout()
    plt.savefig(fname)
    plt.show()
    print('Insight: Bar chart saved to', fname)

# Pie chart for top categories (if exists)
if cat_cols:
    col = cat_cols[0]
    counts = df_clean[col].value_counts().nlargest(6)
    plt.figure(figsize=(6, 6))
    counts.plot.pie(autopct='%1.1f%%')
    plt.title(f'Pie chart of top categories in {col}')
    fname = f'visuals/pie_{col}.png'
    plt.ylabel('')
    plt.savefig(fname)
    plt.show()
    print('Insight: Pie chart saved to', fname)

# Histogram for the first numeric column
if num_cols:
    col = num_cols[0]
    plt.figure(figsize=(8, 4))
    sns.histplot(df_clean[col], bins=30, kde=True, color='steelblue')
    plt.title(f'Histogram of {col}')
    plt.xlabel(col)
    fname = f'visuals/hist_{col}.png'
    plt.savefig(fname)
    plt.show()
    print('Insight: Histogram saved to', fname)

# Scatter plot between two numeric features (if available)
if len(num_cols) >= 2:
    x, y = num_cols[0], num_cols[1]
    plt.figure(figsize=(7, 5))
    sns.scatterplot(x=df_clean[x], y=df_clean[y])
    plt.title(f'Scatter plot: {x} vs {y}')
    plt.xlabel(x)
    plt.ylabel(y)
    fname = f'visuals/scatter_{x}_vs_{y}.png'
    plt.savefig(fname)
    plt.show()
    print('Insight: Scatter plot saved to', fname)

# Box plot for a numeric column grouped by a categorical column (if possible)
if num_cols and cat_cols:
    ncol = num_cols[0]
    ccol = cat_cols[0]
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=df_clean[ccol], y=df_clean[ncol])
    plt.title(f'Box plot of {ncol} by {ccol}')
    plt.xticks(rotation=45)
    fname = f'visuals/box_{ncol}_by_{ccol}.png'
    plt.tight_layout()
    plt.savefig(fname)
    plt.show()
    print('Insight: Box plot saved to', fname)

# Heatmap of correlations already created in EDA; save an example heatmap if numeric columns exist
if len(num_cols) >= 2:
    plt.figure(figsize=(10, 8))
    sns.heatmap(df_clean[num_cols].corr(), annot=True, cmap='coolwarm')
    plt.title('Correlation heatmap (saved)')
    fname = 'visuals/heatmap_correlations.png'
    plt.tight_layout()
    plt.savefig(fname)
    plt.show()
    print('Insight: Heatmap saved to', fname)

## 5) Predictive Model (Baseline Classification)
A simple, beginner-friendly classification workflow using scikit-learn. The notebook attempts to auto-detect a target column but you can set `TARGET` manually.

In [ ]:
# Select target column: try common names then fallback to last column
candidates = ['target', 'label', 'churn', 'outcome']
TARGET = None
for c in candidates:
    if c in df_clean.columns:
        TARGET = c
        break
if TARGET is None:
    TARGET = df_clean.columns[-1]  # fallback to last column

print('Using target column:', TARGET)

# Prepare feature matrix X and target y
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

# If target is numeric with many unique values, convert to categorical bins for classification
if pd.api.types.is_numeric_dtype(y) and y.nunique() > 10:
    y = pd.qcut(y, q=3, labels=False)
    print('Binned numeric target into 3 classes for classification')

# Encode categorical features using one-hot encoding
X = pd.get_dummies(X, drop_first=True)

# Align X and handle any remaining missing values
X = X.fillna(X.median())

# Feature selection: simple variance thresholding (drop cols with zero variance)
keep_cols = X.loc[:, X.nunique() > 1].columns.tolist()
X = X[keep_cols]
print('Final feature count:', X.shape[1])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y if y.nunique()>1 else None)

# Model training: Random Forest (baseline)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Prediction and evaluation
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.4f}')

# Confusion matrix and classification report
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(cm)
print('
Classification Report:')
print(classification_report(y_test, y_pred))

---
Notebook complete.
Next steps: tune model, perform cross-validation, and document findings in the README.